# Introdução

Aqui vamos explorar os arquivos relacionados a migração internacional no Brasil. Usaremos os dados do Sismigra dos anos de 2024, 2025 e 2026.
Deixaremos o sismigra_ativos.csv fora desta primeira análise, porque ele não possui município de residência.

In [1]:
# Preparação 

import pandas as pd 
from pathlib import Path

# Pasta onde estão os arquivos CSV
pasta = Path("../data/raw/migracao")

# Verificar os arquivos disponíveis na pasta
for arquivo in pasta.glob("*.csv"):
    print(f"Arquivo: {arquivo.name}, Tamanho: {round(arquivo.stat().st_size / 1024**2, 2)} MB")

Arquivo: sismigra_2024.csv, Tamanho: 55.28 MB
Arquivo: sismigra_ativos.csv, Tamanho: 365.85 MB
Arquivo: sismigra_janeiro_agosto_2026.csv, Tamanho: 36.61 MB
Arquivo: sismigra_janeiro_dezembro_2025.csv, Tamanho: 58.4 MB


In [2]:
# Carregar os dados dos três arquivos CSV

df_2024 = pd.read_csv(
    pasta / "sismigra_2024.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

df_2025 = pd.read_csv(
    pasta / "sismigra_janeiro_dezembro_2025.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

df_2026 = pd.read_csv(
    pasta / "sismigra_janeiro_agosto_2026.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

print("Arquivos carregados com sucesso!")

Arquivos carregados com sucesso!


In [3]:
# Conferindo a quantidade de linhas e colunas de cada DataFrame
print(f"2024: {df_2024.shape[0]} linhas, {df_2024.shape[1]} colunas")  
print(f"2025: {df_2025.shape[0]} linhas, {df_2025.shape[1]} colunas")  
print(f"2026: {df_2026.shape[0]} linhas, {df_2026.shape[1]} colunas")  


2024: 248167 linhas, 17 colunas
2025: 265523 linhas, 17 colunas
2026: 176896 linhas, 17 colunas


In [4]:
# Verificando se as colunas são iguais entre os DataFrames

print(
    "2024 e 2025:",
    list(df_2024.columns) == list(df_2025.columns)
)

print(
    "2025 e 2026:",
    list(df_2025.columns) == list(df_2026.columns)
)


2024 e 2025: True
2025 e 2026: False


In [5]:
# Valores de faixa etária em 2025
print("2025")
print(df_2025["FAIXA_ETARIA"].value_counts().head(20))

# Valores de idade em 2026
print("\n2024")
print(df_2024["FAIXA_ETARIA"].value_counts().head(20))

2025
FAIXA_ETARIA
25 |-- 40 anos      73739
15 |-- 25 anos      56929
40 |-- 65 anos      45818
NÃO ESPECIFICADO    41344
0 |-- 15 anos       40441
65 |-- anos          7252
Name: count, dtype: int64

2024
FAIXA_ETARIA
25 |-- 40 anos      69532
15 |-- 25 anos      55399
40 |-- 65 anos      44155
0 |-- 15 anos       43073
NÃO ESPECIFICADO    24674
65 |-- anos         11334
Name: count, dtype: int64


### Padronizando as colunas de faixa etária

In [6]:
# Transformando as idades 2026 em faixa etária para padronizar com os outros anos

df_2026["FAIXA_ETARIA"] = pd.cut(
    df_2026["IDADE"],
    bins=[0, 15, 25, 40, 65, float("inf")],
    labels=["0-14", "15-24", "25-39", "40-64", "65+"],
    right=False
)

 # Permite que a string "Não especificado" seja aceita nesta coluna
df_2026["FAIXA_ETARIA"] = df_2026["FAIXA_ETARIA"].cat.add_categories("Não especificado")

# Substitui os valores nulos com sucesso
df_2026["FAIXA_ETARIA"] = df_2026["FAIXA_ETARIA"].fillna("Não especificado")


# Conferindo o resultado da transformação
print(df_2026["FAIXA_ETARIA"].value_counts(dropna=False))


FAIXA_ETARIA
25-39               50917
15-24               38095
Não especificado    30924
40-64               28951
0-14                23348
65+                  4661
Name: count, dtype: int64


In [7]:
# Padronizando a coluna de faixa etária para os DataFrames de 2024 e 2025, para que fiquem iguais à do DataFrame de 2026


# Dicionário de equivalências
faixas = {
    "0 |-- 15 anos": "0-14",
    "15 |-- 25 anos": "15-24",
    "25 |-- 40 anos": "25-39",
    "40 |-- 65 anos": "40-64",
    "65 |-- anos": "65+",
    "NÃO ESPECIFICADO": "Não especificado"
}

# Padronizar as duas tabelas
df_2024["FAIXA_ETARIA"] = df_2024["FAIXA_ETARIA"].replace(faixas)
df_2025["FAIXA_ETARIA"] = df_2025["FAIXA_ETARIA"].replace(faixas)


### Analisando os itens ausentes

In [8]:
ausentes = df_2025.isna().sum()

display(ausentes)

DATAREGISTRO                  0
AMPARO                        0
AMPARO_DESCRICAO              0
CLASSIFICACAO_REGISTRO        0
DATAENTRADA               11312
UNIDADE_FED_RESIDENCIA     1962
MUNICIPIO_RESIDENCIA          0
PAIS_NASCIMENTO               0
PAIS_NACIONALIDADE            1
CONTINENTE                    0
FAIXA_ETARIA                  0
SEXO                          6
ESTADO_CIVIL                  3
PROFISSAO                     0
STATUS                        0
CLASSIFICACAO_REVISADA        0
TIPOLOGIA_AMPAROS             0
dtype: int64

In [9]:
ausentes = df_2026.isna().sum()

display(ausentes)

DATAREGISTRO                  0
AMPARO                        0
AMPARO_DESCRICAO              0
CLASSIFICACAO_REGISTRO        0
DATAENTRADA                   0
UNIDADE_FED_RESIDENCIA     1710
MUNICIPIO_RESIDENCIA          0
PAIS_NASCIMENTO               0
PAIS_NACIONALIDADE            2
CONTINENTE                    0
IDADE                     30302
SEXO                          0
ESTADO_CIVIL                  0
PROFISSAO                     0
STATUS                        0
CLASSIFICACAO_REVISADA        0
TIPOLOGIA_AMPAROS             0
FAIXA_ETARIA                  0
dtype: int64

DATAENTRADA: Os 11.312 registros sem data de entrada representam aproximadamente 4,26% da base. Porém, como nosso principal indicador é o número de registros, vamos manter esses registros sem data de entrata. 
UNIDADE_FED_RESIDENCIA: Os percentuais são relativamente baixos: 0,74% em 2025 e 0,97% em 2026. Porém, essa variável é fundamental para nosso projeto. Em ambos os DataFrames, 100% dos registros onde UNIDADE_FED_RESIDENCIA é null os municípios constam como "Não especificado". 

In [10]:
# Expcluindo as linhas onde não há informações geográficas (UNIDADE_FED_RESIDENCIA). 1.962 registros em 2025 e 1.710 em 2026.
df_2025 = df_2025.dropna(subset=["UNIDADE_FED_RESIDENCIA"])
df_2026 = df_2026.dropna(subset=["UNIDADE_FED_RESIDENCIA"])

In [11]:
negativas = df_2026[df_2026["IDADE"] < 0]

print("Total de idades negativas:", len(negativas))

print(
    "Entre -2026 e -1900:",
    negativas["IDADE"].between(-2026, -1900).sum()
)

print(
    "Fora desse intervalo:",
    (~negativas["IDADE"].between(-2026, -1900)).sum()
)


Total de idades negativas: 615
Entre -2026 e -1900: 369
Fora desse intervalo: 246


A base SISMIGRA 2026 apresenta 622 registros (0,35%) com idades negativas, considerados inválidos para análise demográfica. Esses registros foram preservados para análises territoriais, mas excluídos dos cálculos que exigem idade válida. A origem dos valores negativos permanece não esclarecida.

In [12]:
negativas[
    ~negativas["IDADE"].between(-2026, -1900)
]["IDADE"].value_counts().head(30)

IDADE
-1.0       77
-2.0        9
-3.0        8
-1799.0     6
-1780.0     5
-4.0        4
-1791.0     4
-1806.0     4
-1777.0     4
-1771.0     3
-1820.0     3
-1790.0     3
-1783.0     3
-1784.0     3
-1800.0     3
-1802.0     3
-1765.0     2
-1782.0     2
-11.0       2
-1793.0     2
-1767.0     2
-1789.0     2
-872.0      2
-879.0      2
-877.0      2
-1812.0     2
-1773.0     2
-1794.0     2
-1813.0     2
-883.0      2
Name: count, dtype: int64

In [13]:
# Marcando as idades negativas como "Não identificado" na coluna de faixa etária

df_2026["FAIXA_ETARIA"] = (
    df_2026["FAIXA_ETARIA"]
    .astype("object")
    .fillna("Não identificado")
)

df_2026["FAIXA_ETARIA"].value_counts(dropna=False)

FAIXA_ETARIA
25-39               50335
15-24               37814
Não especificado    30866
40-64               28371
0-14                23242
65+                  4558
Name: count, dtype: int64

### Verificando as duplicidades


In [14]:
for ano, df in [(2024, df_2024), (2025, df_2025), (2026, df_2026)]:
    print(ano, ":", df.duplicated().sum(), "duplicatas")

2024 : 149872 duplicatas
2025 : 157552 duplicatas
2026 : 59893 duplicatas


Vamos visualizar alguns registros repetidos de 2025:

In [15]:
duplicatas = df_2025[df_2025.duplicated(keep=False)]

display(duplicatas.head(10))

,DATAREGISTRO,AMPARO,AMPARO_DESCRICAO,CLASSIFICACAO_REGISTRO,DATAENTRADA,UNIDADE_FED_RESIDENCIA,MUNICIPIO_RESIDENCIA,PAIS_NASCIMENTO,PAIS_NACIONALIDADE,CONTINENTE,FAIXA_ETARIA,SEXO,ESTADO_CIVIL,PROFISSAO,STATUS,CLASSIFICACAO_REVISADA,TIPOLOGIA_AMPAROS
5,2025,36,36 - ART.23 L.13445/17 C/C ART.90 DEC.9199/17,Fronteiriço,2025,AC,EPITACIOLÂNDIA,BOLÍVIA,BOLÍVIA,AMÉRICA DO SUL,0-14,Feminino,SOLTEIRO,"MENOR ( CRIANCA , NAO ESTUDANTE )",Ativo,FRONTEIRIÇO,FRONTEIRIÇOS
6,2025,36,36 - ART.23 L.13445/17 C/C ART.90 DEC.9199/17,Fronteiriço,2025,AC,EPITACIOLÂNDIA,BOLÍVIA,BOLÍVIA,AMÉRICA DO SUL,0-14,Feminino,SOLTEIRO,"MENOR ( CRIANCA , NAO ESTUDANTE )",Ativo,FRONTEIRIÇO,FRONTEIRIÇOS
18,2025,256,256 - FRONTEIRICO - PORTARIA MJ 1512/2014,Fronteiriço,2025,-,NÃO ESPECIFICADO,FRANÇA,FRANÇA,EUROPA,40-64,Masculino,SOLTEIRO,"DIRETOR , GERENTE OU PROPRIETARIO, DE INDUSTRI...",Ativo,FRONTEIRIÇO,FRONTEIRIÇOS
19,2025,256,256 - FRONTEIRICO - PORTARIA MJ 1512/2014,Fronteiriço,2025,-,NÃO ESPECIFICADO,FRANÇA,FRANÇA,EUROPA,40-64,Masculino,SOLTEIRO,"DIRETOR , GERENTE OU PROPRIETARIO, DE INDUSTRI...",Ativo,FRONTEIRIÇO,FRONTEIRIÇOS
22,2025,36,36 - ART.23 L.13445/17 C/C ART.90 DEC.9199/17,Fronteiriço,2024,PR,FOZ DO IGUAÇU,PARAGUAI,PARAGUAI,AMÉRICA DO SUL,25-39,Feminino,SOLTEIRO,"ECONOMISTA, ATUARIO, CONTADOR, TECNICO EM CONT...",Ativo,FRONTEIRIÇO,FRONTEIRIÇOS
23,2025,36,36 - ART.23 L.13445/17 C/C ART.90 DEC.9199/17,Fronteiriço,2024,PR,FOZ DO IGUAÇU,PARAGUAI,PARAGUAI,AMÉRICA DO SUL,25-39,Feminino,SOLTEIRO,"ECONOMISTA, ATUARIO, CONTADOR, TECNICO EM CONT...",Ativo,FRONTEIRIÇO,FRONTEIRIÇOS
28,2025,36,36 - ART.23 L.13445/17 C/C ART.90 DEC.9199/17,Fronteiriço,2024,PR,FOZ DO IGUAÇU,PARAGUAI,PARAGUAI,AMÉRICA DO SUL,40-64,Feminino,DESCONHECIDO,OUTRA OCUPACAO NAO CLASSIFICADA,Ativo,FRONTEIRIÇO,FRONTEIRIÇOS
32,2025,36,36 - ART.23 L.13445/17 C/C ART.90 DEC.9199/17,Fronteiriço,2025,PR,FOZ DO IGUAÇU,PARAGUAI,PARAGUAI,AMÉRICA DO SUL,40-64,Feminino,VIUVO,SEM OCUPACAO,Ativo,FRONTEIRIÇO,FRONTEIRIÇOS
34,2025,36,36 - ART.23 L.13445/17 C/C ART.90 DEC.9199/17,Fronteiriço,2025,PR,FOZ DO IGUAÇU,PARAGUAI,PARAGUAI,AMÉRICA DO SUL,25-39,Feminino,SOLTEIRO,SEM OCUPACAO,Ativo,FRONTEIRIÇO,FRONTEIRIÇOS
41,2025,36,36 - ART.23 L.13445/17 C/C ART.90 DEC.9199/17,Fronteiriço,2024,PR,FOZ DO IGUAÇU,PARAGUAI,PARAGUAI,AMÉRICA DO SUL,40-64,Masculino,SOLTEIRO,SEM OCUPACAO,Ativo,FRONTEIRIÇO,FRONTEIRIÇOS


In [16]:
# Verificando a quantidade de registros com "MUNICIPIO_RESIDENCIA" como "NÃO ESPECIFICADO" em cada DataFrame
for ano, df in [(2024, df_2024), (2025, df_2025), (2026, df_2026)]:
    print(
        ano,
        (df["MUNICIPIO_RESIDENCIA"] == "NÃO ESPECIFICADO").sum()
    )

2024 25555
2025 26766
2026 22075


In [17]:
for ano, df in [(2024, df_2024), (2025, df_2025), (2026, df_2026)]:
    print(f"\n{ano}")
    print(df["MUNICIPIO_RESIDENCIA"].value_counts(dropna=False).tail(15))


2024
MUNICIPIO_RESIDENCIA
PARDINHO                   10
SAPEZAL                    10
URUBICI                    10
CANELINHA                  10
SIQUEIRA CAMPOS            10
NOVO HORIZONTE DO NORTE    10
TUNÁPOLIS                  10
IBIAÇÁ                     10
CURVELO                    10
BELO ORIENTE               10
ILHA SOLTEIRA              10
BARÃO                      10
NOVA SANTA ROSA            10
ESPERANÇA DO SUL           10
AQUIRAZ                    10
Name: count, dtype: int64

2025
MUNICIPIO_RESIDENCIA
GUAPIAÇU                     10
RANCHO QUEIMADO              10
SÃO MIGUEL                   10
CHIAPETTA                    10
SÃO JOSÉ DO HORTÊNCIO        10
SARZEDO                      10
CRISSIUMAL                   10
TRÊS CACHOEIRAS              10
SANTA LUZIA                  10
SANTO ANTÔNIO DO LEVERGER    10
COLATINA                     10
MIRASSOL DOESTE              10
NOVA MONTE VERDE             10
ESTRELA DOESTE               10
ACEGUA                

In [18]:
for ano, df in [(2024, df_2024), (2025, df_2025), (2026, df_2026)]:
    invalidos = df[
        df["MUNICIPIO_RESIDENCIA"].astype("string").str.strip().isin(
            ["NÃO ESPECIFICADO", "-", ""]
        ) | df["MUNICIPIO_RESIDENCIA"].isna()
    ]

    print(f"\n{ano}")
    print(invalidos["MUNICIPIO_RESIDENCIA"].value_counts(dropna=False))


2024
MUNICIPIO_RESIDENCIA
NÃO ESPECIFICADO    25555
Name: count, dtype: int64

2025
MUNICIPIO_RESIDENCIA
NÃO ESPECIFICADO    26766
Name: count, dtype: int64

2026
MUNICIPIO_RESIDENCIA
NÃO ESPECIFICADO    22075
Name: count, dtype: int64


Confirmamos que NÃO ESPECIFICADO é o único valor inválido encontrado na coluna de município dos três anos.

Vamos excluir esses registros dos DataFrames utilizados na análise territorial.

In [19]:
for df in [df_2024, df_2025, df_2026]:
    df.drop(
        df[df["MUNICIPIO_RESIDENCIA"] == "NÃO ESPECIFICADO"].index,
        inplace=True
    )

print("2024:", len(df_2024))
print("2025:", len(df_2025))
print("2026:", len(df_2026))

2024: 222612
2025: 236795
2026: 153111


In [20]:
# Verificando que as colunas "UNIDADE_FED_RESIDENCIA" estão preenchidas corretamente após a exclusão dos registros inválidos

for df in [df_2024, df_2025, df_2026]:
    print(df["UNIDADE_FED_RESIDENCIA"].value_counts().head(27))

UNIDADE_FED_RESIDENCIA
SP    50356
RR    44185
PR    29592
SC    29201
RS    15983
AM    11619
RJ     9070
MS     5563
MG     5153
MT     4922
GO     2891
DF     2478
RO     2371
BA     2039
CE     1311
PA      978
PE      966
AC      849
RN      636
ES      601
AP      415
PB      414
MA      343
AL      259
PI      207
SE      141
TO       69
Name: count, dtype: int64
UNIDADE_FED_RESIDENCIA
SP    55573
RR    38896
PR    34991
SC    32616
RS    17376
RJ    11016
AM     9531
MG     6048
MT     5697
MS     5316
GO     3756
DF     2558
BA     2310
RO     2173
CE     1412
PE     1339
AC     1234
PA      958
RN      952
ES      882
PB      628
AP      399
MA      330
AL      228
SE      222
TO      186
PI      168
Name: count, dtype: int64
UNIDADE_FED_RESIDENCIA
SP    37914
RR    23506
SC    21970
PR    21465
RS    11885
RJ     7263
AM     5332
MG     3895
MT     3690
MS     3117
GO     2787
BA     2472
DF     1545
RO     1183
PE      865
CE      708
ES      653
RN      621
PB      500
AC 

In [21]:
# Verificando que a coluna de município corresponde à unidade federativa na qual o município está localizado

df_2025.groupby(
    ["UNIDADE_FED_RESIDENCIA", "MUNICIPIO_RESIDENCIA"]
).size().sort_values(ascending=False).head(20)

UNIDADE_FED_RESIDENCIA  MUNICIPIO_RESIDENCIA
SP                      SÃO PAULO               38523
RR                      BOA VISTA               35014
PR                      CURITIBA                12861
AM                      MANAUS                   8759
RJ                      RIO DE JANEIRO           7426
SC                      FLORIANÓPOLIS            4508
                        CHAPECÓ                  4399
                        JOINVILLE                3492
PR                      FOZ DO IGUAÇU            3325
                        CASCAVEL                 3143
RS                      CAXIAS DO SUL            2709
DF                      BRASÍLIA                 2558
MT                      CUIABÁ                   2441
RS                      PORTO ALEGRE             2355
RR                      PACARAIMA                2169
MG                      BELO HORIZONTE           1920
SP                      GUARULHOS                1902
                        CAMPINAS     

In [22]:
for df, ano in [(df_2024, 2024), (df_2025, 2025), (df_2026, 2026)]:
    print(f"\n{ano}")
    print(
        df.shape)


2024
(222612, 17)

2025
(236795, 17)

2026
(153111, 18)


### Padronizando as informações geográficas usando o código do IBGE para municípios e estados

In [23]:
# Verificando se existem municípios com o mesmo nome em estados diferentes

municipios = df_2025.groupby("MUNICIPIO_RESIDENCIA")["UNIDADE_FED_RESIDENCIA"].nunique()

print("Municípios com mais de uma unidade federativa:", (municipios > 1).sum())

Municípios com mais de uma unidade federativa: 5


In [24]:
print(municipios[municipios > 1])

MUNICIPIO_RESIDENCIA
ITAPIRANGA     2
PALMAS         2
PINHALZINHO    2
SARANDI        2
SÃO CARLOS     2
Name: UNIDADE_FED_RESIDENCIA, dtype: int64


In [25]:
# Usando a API de localidade do IBGE para associar os municípios aos seus respectivos estados, caso haja inconsistências nos dados

import pandas as pd

url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

ibge = pd.read_json(url)

display(ibge.head())


# Essa tabela será nossa referência para padronizar os municípios e integrar as bases do SISMIGRA, CAGED e INEP.

,id,nome,microrregiao,regiao-imediata
0,1100015,Alta Floresta D'Oeste,"{'id': 11006, 'nome': 'Cacoal', 'mesorregiao':...","{'id': 110005, 'nome': 'Cacoal', 'regiao-inter..."
1,1100023,Ariquemes,"{'id': 11003, 'nome': 'Ariquemes', 'mesorregia...","{'id': 110002, 'nome': 'Ariquemes', 'regiao-in..."
2,1100031,Cabixi,"{'id': 11008, 'nome': 'Colorado do Oeste', 'me...","{'id': 110006, 'nome': 'Vilhena', 'regiao-inte..."
3,1100049,Cacoal,"{'id': 11006, 'nome': 'Cacoal', 'mesorregiao':...","{'id': 110005, 'nome': 'Cacoal', 'regiao-inter..."
4,1100056,Cerejeiras,"{'id': 11008, 'nome': 'Colorado do Oeste', 'me...","{'id': 110006, 'nome': 'Vilhena', 'regiao-inte..."


In [26]:
# Organizando a tabela do IBGE para facilitar a integração com os dados do SISMIGRA

ibge_municipios = pd.DataFrame({
    "codigo_ibge": ibge["id"],
    "municipio": ibge["nome"].str.upper(),
    "uf": ibge["regiao-imediata"].apply(
        lambda x: x["regiao-intermediaria"]["UF"]["sigla"]
    )
})

display(ibge_municipios.head())

,codigo_ibge,municipio,uf
0,1100015,ALTA FLORESTA D'OESTE,RO
1,1100023,ARIQUEMES,RO
2,1100031,CABIXI,RO
3,1100049,CACOAL,RO
4,1100056,CEREJEIRAS,RO


In [27]:
# Verificando valores ausentes ou duplicados na tabela do IBGE

print("Total de municípios:", len(ibge_municipios))

print("\nValores ausentes:")
print(ibge_municipios.isna().sum())

print("\nCódigos IBGE duplicados:")
print(ibge_municipios["codigo_ibge"].duplicated().sum())

Total de municípios: 5571

Valores ausentes:
codigo_ibge    0
municipio      0
uf             0
dtype: int64

Códigos IBGE duplicados:
0


### Cruzando os dados das tabelas Sismigra com a tabela de municípios do IBGE

##### Começando pela tabela de 2025: 

In [28]:
# Verificando se todos os municípios do SISMIGRA 2025 estão presentes na tabela do IBGE

df_2025 = df_2025.merge(
    ibge_municipios, 
    left_on=["MUNICIPIO_RESIDENCIA", "UNIDADE_FED_RESIDENCIA"],
    right_on=["municipio", "uf"],
    how="left"
)

print("Registros totais:", len(df_2025))
print("Com código IBGE:", df_2025["codigo_ibge"].notna().sum())
print("Sem código IBGE:", df_2025["codigo_ibge"].isna().sum())

Registros totais: 236795
Com código IBGE: 235783
Sem código IBGE: 1012


In [29]:
# Identificando os municípios do SISMIGRA 2025 que não possuem correspondência na tabela do IBGE

df_2025[df_2025["codigo_ibge"].isna()].groupby(["UNIDADE_FED_RESIDENCIA", "MUNICIPIO_RESIDENCIA"]).size().sort_values(ascending=False)

UNIDADE_FED_RESIDENCIA  MUNICIPIO_RESIDENCIA     
RS                      SANTANA DO LIVRAMENTO        489
SC                      HERVAL DOESTE                137
RJ                      PARATI                       107
SP                      SANTA BÁRBARA DOESTE          86
RR                      SÃO LUIZ                      63
PR                      ITAPEJARA DOESTE              34
CE                      ACARAPÉ                       32
MT                      JUREMA                        18
RO                      ESPIGÃO DOESTE                14
MT                      SANTO ANTÔNIO DO LEVERGER     10
                        MIRASSOL DOESTE               10
SP                      ESTRELA DOESTE                10
RS                      ACEGUA                         2
dtype: int64

In [30]:
ibge_municipios[
    ibge_municipios["municipio"].str.contains(
        "LIVRAMENTO|HERVAL|PARAT|BÁRBARA",
        case=False,
        na=False
    )
][["codigo_ibge", "municipio", "uf"]]

,codigo_ibge,municipio,uf
257,1506351,SANTA BÁRBARA DO PARÁ,PA
1347,2508505,LIVRAMENTO,PB
1688,2703601,JAPARATINGA,AL
1782,2803302,JAPARATUBA,SE
2060,2919504,LIVRAMENTO DE NOSSA SENHORA,BA
2117,2923704,PARATINGA,BA
2163,2927507,SANTA BÁRBARA,BA
2907,3157203,SANTA BÁRBARA,MG
2908,3157252,SANTA BÁRBARA DO LESTE,MG
2909,3157278,SANTA BÁRBARA DO MONTE VERDE,MG


In [31]:
# Corrigindo os 4 municípios mais problemáticos no DataFrame do SISMIGRA 2025, para que correspondam aos municípios corretos na tabela do IBGE

correcoes = {
    "SANTANA DO LIVRAMENTO": "SANT'ANA DO LIVRAMENTO",
    "HERVAL DOESTE": "HERVAL D'OESTE",
    "PARATI": "PARATY",
    "SANTA BÁRBARA DOESTE": "SANTA BÁRBARA D'OESTE"
}

df_2025["MUNICIPIO_PADRONIZADO"] = (
    df_2025["MUNICIPIO_RESIDENCIA"].replace(correcoes)
)

df_2025[
    ["MUNICIPIO_RESIDENCIA", "MUNICIPIO_PADRONIZADO"]
].drop_duplicates().query(
    "MUNICIPIO_RESIDENCIA != MUNICIPIO_PADRONIZADO"
)


,MUNICIPIO_RESIDENCIA,MUNICIPIO_PADRONIZADO
67,SANTANA DO LIVRAMENTO,SANT'ANA DO LIVRAMENTO
4278,HERVAL DOESTE,HERVAL D'OESTE
5696,PARATI,PARATY
38455,SANTA BÁRBARA DOESTE,SANTA BÁRBARA D'OESTE


In [32]:
# Buscando nomes semelhantes para os municípios restantes que não possuem correspondência na tabela do IBGE

import difflib

pendentes = [
    "SÃO LUIZ", "ITAPEJARA DOESTE", "ACARAPÉ",
    "JUREMA", "ESPIGÃO DOESTE",
    "SANTO ANTÔNIO DO LEVERGER",
    "MIRASSOL DOESTE", "ESTRELA DOESTE", "ACEGUA"
]

for nome in pendentes:
    semelhantes = difflib.get_close_matches(
        nome,
        ibge_municipios["municipio"],
        n=3,
        cutoff=0.6
    )
    print(nome, "→", semelhantes)

SÃO LUIZ → ['SÃO LUÍS', 'SÃO FÉLIX', 'SÃO LUIZ GONZAGA']
ITAPEJARA DOESTE → ["ITAPEJARA D'OESTE", "APARECIDA D'OESTE", 'ITAPUÃ DO OESTE']
ACARAPÉ → ['ACARAPE', 'ACARAÚ', 'VACARIA']
JUREMA → ['JUREMA', 'JUREMA', 'URUPEMA']
ESPIGÃO DOESTE → ["ESPIGÃO D'OESTE", 'UNIÃO DO OESTE', 'RIO DO OESTE']
SANTO ANTÔNIO DO LEVERGER → ['SANTO ANTÔNIO DE LEVERGER', 'SANTO ANTÔNIO DO LESTE', 'SANTO ANTÔNIO DO RETIRO']
MIRASSOL DOESTE → ["MIRASSOL D'OESTE", "PALMEIRA D'OESTE", 'LIMEIRA DO OESTE']
ESTRELA DOESTE → ["ESTRELA D'OESTE", 'ESTRELA DO NORTE', 'ESTRELA DO NORTE']
ACEGUA → ['ACEGUÁ', 'AGUAÍ', 'TAGUAÍ']


In [33]:
# Confirmando os UFs no IBGE

nomes = [
    "SÃO LUIZ", "JUREMA", "ACARAPE",
    "ITAPEJARA D'OESTE", "ESPIGÃO D'OESTE",
    "SANTO ANTÔNIO DE LEVERGER",
    "MIRASSOL D'OESTE", "ESTRELA D'OESTE", "ACEGUÁ"
]

display(
    ibge_municipios[
        ibge_municipios["municipio"].isin(nomes)
    ][["municipio", "uf", "codigo_ibge"]]
)

,municipio,uf,codigo_ibge
8,ESPIGÃO D'OESTE,RO,1100098
777,JUREMA,PI,2205532
892,ACARAPE,CE,2300150
1560,JUREMA,PE,2608404
3440,ESTRELA D'OESTE,SP,3515202
4070,ITAPEJARA D'OESTE,PR,4111209
4606,ACEGUÁ,RS,4300034
5250,MIRASSOL D'OESTE,MT,5105622
5303,SANTO ANTÔNIO DE LEVERGER,MT,5107800


In [34]:
# Aplicando correções para 7 municípios

correcoes.update({
    "ITAPEJARA DOESTE": "ITAPEJARA D'OESTE",
    "ACARAPÉ": "ACARAPE",
    "ESPIGÃO DOESTE": "ESPIGÃO D'OESTE",
    "SANTO ANTÔNIO DO LEVERGER": "SANTO ANTÔNIO DE LEVERGER",
    "MIRASSOL DOESTE": "MIRASSOL D'OESTE",
    "ESTRELA DOESTE": "ESTRELA D'OESTE",
    "ACEGUA": "ACEGUÁ"
})

df_2025["MUNICIPIO_PADRONIZADO"] = (
    df_2025["MUNICIPIO_RESIDENCIA"].replace(correcoes)
)

In [35]:
# Refazendo o cruzamento como IBGE para verificar se todos os municípios do SISMIGRA 2025 agora possuem correspondência na tabela do IBGE

df_2025 = df_2025.drop(
    columns=["codigo_ibge", "municipio", "uf"],
    errors="ignore"
)

df_2025 = df_2025.merge(
    ibge_municipios,
    left_on=["MUNICIPIO_PADRONIZADO", "UNIDADE_FED_RESIDENCIA"],
    right_on=["municipio", "uf"],
    how="left"
)

print("Com código IBGE:", df_2025["codigo_ibge"].notna().sum())
print("Sem código IBGE:", df_2025["codigo_ibge"].isna().sum())

Com código IBGE: 236714
Sem código IBGE: 81


In [36]:
df_2025[df_2025["codigo_ibge"].isna()].groupby(
    ["UNIDADE_FED_RESIDENCIA", "MUNICIPIO_RESIDENCIA"]
).size()

UNIDADE_FED_RESIDENCIA  MUNICIPIO_RESIDENCIA
MT                      JUREMA                  18
RR                      SÃO LUIZ                63
dtype: int64

In [37]:
# Corrigindo o município de São Luiz
df_2025.loc[
    (df_2025["MUNICIPIO_RESIDENCIA"] == "SÃO LUIZ") &
    (df_2025["UNIDADE_FED_RESIDENCIA"] == "RR"),
    "codigo_ibge"
] = 1400605

print("Sem código IBGE:", df_2025["codigo_ibge"].isna().sum())

Sem código IBGE: 18


Agora temos 236.777 registros de 2025 com código IBGE, restando apenas 18 sem correspondência (JUREMA, MT).

Vamos manter esses 18 registros sem código, documentando a inconsistência.

In [38]:
# Aplicando as mesmas correções para os DataFrames de 2024 e 2026, para padronizar os nomes dos municípios

for df in [df_2024, df_2026]:
    df["MUNICIPIO_PADRONIZADO"] = (
        df["MUNICIPIO_RESIDENCIA"].replace(correcoes)
    )

In [39]:
for df in [df_2024, df_2026]:
    df["codigo_ibge"] = df.merge(
        ibge_municipios,
        left_on=["MUNICIPIO_PADRONIZADO", "UNIDADE_FED_RESIDENCIA"],
        right_on=["municipio", "uf"],
        how="left"
    )["codigo_ibge"].values

print("2024 sem código IBGE:", df_2024["codigo_ibge"].isna().sum())
print("2026 sem código IBGE:", df_2026["codigo_ibge"].isna().sum())

2024 sem código IBGE: 242
2026 sem código IBGE: 87


In [40]:
for ano, df in [(2024, df_2024), (2026, df_2026)]:
    print(f"\n{ano}")

    pendentes = df[df["codigo_ibge"].isna()]

    print(
        pendentes.groupby(
            ["UNIDADE_FED_RESIDENCIA", "MUNICIPIO_RESIDENCIA"]
        ).size().sort_values(ascending=False)
    )


2024
UNIDADE_FED_RESIDENCIA  MUNICIPIO_RESIDENCIA
RR                      SÃO LUIZ                125
SP                      EMBU                     79
MT                      JUREMA                   38
dtype: int64

2026
UNIDADE_FED_RESIDENCIA  MUNICIPIO_RESIDENCIA
RR                      SÃO LUIZ                40
MT                      JUREMA                  33
RO                      ALTA FLORESTA DOESTE    14
dtype: int64


In [41]:
# Corrigir nomes no dicionário
correcoes.update({
    "EMBU": "EMBU DAS ARTES",
    "ALTA FLORESTA DOESTE": "ALTA FLORESTA D'OESTE"
})

# Atualizar os nomes padronizados
for df in [df_2024, df_2026]:
    df["MUNICIPIO_PADRONIZADO"] = (
        df["MUNICIPIO_RESIDENCIA"].replace(correcoes)
    )

    # Corrigir diretamente os códigos IBGE
    df.loc[
        df["MUNICIPIO_RESIDENCIA"] == "EMBU",
        "codigo_ibge"
    ] = 3515004

    df.loc[
        df["MUNICIPIO_RESIDENCIA"] == "ALTA FLORESTA DOESTE",
        "codigo_ibge"
    ] = 1100015

    df.loc[
        (df["MUNICIPIO_RESIDENCIA"] == "SÃO LUIZ") &
        (df["UNIDADE_FED_RESIDENCIA"] == "RR"),
        "codigo_ibge"
    ] = 1400605

print("2024 sem código:", df_2024["codigo_ibge"].isna().sum())
print("2026 sem código:", df_2026["codigo_ibge"].isna().sum())

2024 sem código: 38
2026 sem código: 33


In [42]:
for ano, df in [(2024, df_2024), (2025, df_2025), (2026, df_2026)]:
    print(f"\n{ano}")
    print("Total de registros:", len(df))
    print("Com código IBGE:", df["codigo_ibge"].notna().sum())
    print("Códigos distintos:", df["codigo_ibge"].nunique())


2024
Total de registros: 222612
Com código IBGE: 222574
Códigos distintos: 747

2025
Total de registros: 236795
Com código IBGE: 236777
Códigos distintos: 798

2026
Total de registros: 153111
Com código IBGE: 153078
Códigos distintos: 650


### Salvando as três bases padronizadas
(Vamos usar o formato Parquet, que costuma ocupar menos espaço e permite leituras mais rápidas do que CSV.)

In [43]:
pasta_destino = Path("../data/processed/migracao")
pasta_destino.mkdir(parents=True, exist_ok=True)

df_2024.to_parquet(pasta_destino / "sismigra_2024.parquet", index=False)
df_2025.to_parquet(pasta_destino / "sismigra_2025.parquet", index=False)
df_2026.to_parquet(pasta_destino / "sismigra_2026.parquet", index=False)

print("Arquivos salvos com sucesso!")

Arquivos salvos com sucesso!


## Construindo indicadores migratórios por município

### Contando registros por município em 2025

In [44]:
migracao_2025 = ( 
    df_2025.dropna(subset=["codigo_ibge"]).groupby(["codigo_ibge", "MUNICIPIO_PADRONIZADO", "UNIDADE_FED_RESIDENCIA"]).size().reset_index(name="total_registros").sort_values(by="total_registros", ascending=False))

display(migracao_2025.head(20))

,codigo_ibge,MUNICIPIO_PADRONIZADO,UNIDADE_FED_RESIDENCIA,total_registros
336,3550308.0,SÃO PAULO,SP,38523
36,1400100.0,BOA VISTA,RR,35014
379,4106902.0,CURITIBA,PR,12861
27,1302603.0,MANAUS,AM,8759
228,3304557.0,RIO DE JANEIRO,RJ,7426
488,4205407.0,FLORIANÓPOLIS,SC,4508
477,4204202.0,CHAPECÓ,SC,4399
518,4209102.0,JOINVILLE,SC,3492
385,4108304.0,FOZ DO IGUAÇU,PR,3325
371,4104808.0,CASCAVEL,PR,3143


In [45]:
# Criando uma tabela comparativa entre os anos de 2024 e 2025

registros_2024 = df_2024.groupby("codigo_ibge").size()
registros_2025 = df_2025.groupby("codigo_ibge").size()

comparacao = pd.DataFrame({
    "2024": registros_2024,
    "2025": registros_2025
}).fillna(0)

comparacao["variacao_%"] = ((comparacao["2025"] - comparacao["2024"]) / comparacao["2024"].replace(0, float("nan")) * 100).round(2)

display(comparacao.head(20))

,2024,2025,variacao_%
codigo_ibge,,,
1100023.0,0.0,17.0,NaN
1100049.0,94.0,89.0,-5.32
1100056.0,10.0,24.0,140.00
1100080.0,120.0,116.0,-3.33
1100098.0,19.0,14.0,-26.32
1100106.0,143.0,130.0,-9.09
1100122.0,118.0,195.0,65.25
1100189.0,16.0,17.0,6.25
1100205.0,1445.0,1150.0,-20.42


In [46]:
# Identificando os municípios com maior crescimento 

crescimento = comparacao[comparacao["2024"] >= 100]

crescimento = crescimento.sort_values(by="variacao_%", ascending=False)

crescimento.head(20)

,2024,2025,variacao_%
codigo_ibge,,,
2919207.0,114.0,433.0,279.82
4201307.0,139.0,287.0,106.47
4117909.0,100.0,204.0,104.00
3548906.0,167.0,340.0,103.59
5201405.0,199.0,382.0,91.96
3523107.0,306.0,584.0,90.85
5106224.0,132.0,251.0,90.15
5005681.0,222.0,420.0,89.19
4311403.0,150.0,280.0,86.67


In [47]:
comparacao = comparacao.reset_index()

comparacao = comparacao.merge(
    ibge_municipios,
    on="codigo_ibge",
    how="left"
)

comparacao = comparacao[
    ["codigo_ibge", "municipio", "uf", "2024", "2025", "variacao_%"]
]

display(comparacao.head(20))

,codigo_ibge,municipio,uf,2024,2025,variacao_%
0,1100023.0,ARIQUEMES,RO,0.0,17.0,NaN
1,1100049.0,CACOAL,RO,94.0,89.0,-5.32
2,1100056.0,CEREJEIRAS,RO,10.0,24.0,140.00
3,1100080.0,COSTA MARQUES,RO,120.0,116.0,-3.33
4,1100098.0,ESPIGÃO D'OESTE,RO,19.0,14.0,-26.32
5,1100106.0,GUAJARÁ-MIRIM,RO,143.0,130.0,-9.09
6,1100122.0,JI-PARANÁ,RO,118.0,195.0,65.25
7,1100189.0,PIMENTA BUENO,RO,16.0,17.0,6.25
8,1100205.0,PORTO VELHO,RO,1445.0,1150.0,-20.42
9,1100288.0,ROLIM DE MOURA,RO,32.0,51.0,59.38


In [48]:
crescimento = comparacao[comparacao["2024"] >= 100]

crescimento = crescimento.sort_values(
    "variacao_%", ascending=False
)

display(crescimento.head(20))

,codigo_ibge,municipio,uf,2024,2025,variacao_%
141,2919207.0,LAURO DE FREITAS,BA,114.0,433.0,279.82
508,4201307.0,ARAQUARI,SC,139.0,287.0,106.47
462,4117909.0,PALOTINA,PR,100.0,204.0,104.00
376,3548906.0,SÃO CARLOS,SP,167.0,340.0,103.59
859,5201405.0,APARECIDA DE GOIÂNIA,GO,199.0,382.0,91.96
325,3523107.0,ITAQUAQUECETUBA,SP,306.0,584.0,90.85
840,5106224.0,NOVA MUTUM,MT,132.0,251.0,90.15
805,5005681.0,MUNDO NOVO,MS,222.0,420.0,89.19
707,4311403.0,LAJEADO,RS,150.0,280.0,86.67
391,3556206.0,VALINHOS,SP,111.0,202.0,81.98


In [49]:
# Adicionando uma coluna com a variação absoluta entre os anos de 2024 e 2025

comparacao["variacao_absoluta"] = (
    comparacao["2025"] - comparacao["2024"]
)

display(
    comparacao.sort_values(
        "variacao_absoluta", ascending=False
    ).head(20)
)

,codigo_ibge,municipio,uf,2024,2025,variacao_%,variacao_absoluta
379,3550308.0,SÃO PAULO,SP,36158.0,38523.0,6.54,2365.0
423,4106902.0,CURITIBA,PR,10507.0,12861.0,22.40,2354.0
263,3304557.0,RIO DE JANEIRO,RJ,6356.0,7426.0,16.83,1070.0
730,4314902.0,PORTO ALEGRE,RS,1725.0,2355.0,36.52,630.0
291,3509502.0,CAMPINAS,SP,1272.0,1894.0,48.90,622.0
576,4209102.0,JOINVILLE,SC,2956.0,3492.0,18.13,536.0
430,4108304.0,FOZ DO IGUAÇU,PR,2895.0,3325.0,14.85,430.0
868,5208707.0,GOIÂNIA,GO,1270.0,1689.0,32.99,419.0
546,4205407.0,FLORIANÓPOLIS,SC,4094.0,4508.0,10.11,414.0
255,3302403.0,MACAÉ,RJ,742.0,1145.0,54.31,403.0


In [50]:
# Adicionando os dados de 2026 à tabela comparativa

registros_2026 = df_2026.groupby("codigo_ibge").size()

comparacao["2026_parcial"] = (
    comparacao["codigo_ibge"].map(registros_2026).fillna(0).astype(int)
)

display(comparacao.head(20))

,codigo_ibge,municipio,uf,2024,2025,variacao_%,variacao_absoluta,2026_parcial
0,1100023.0,ARIQUEMES,RO,0.0,17.0,NaN,17.0,0
1,1100049.0,CACOAL,RO,94.0,89.0,-5.32,-5.0,34
2,1100056.0,CEREJEIRAS,RO,10.0,24.0,140.00,14.0,20
3,1100080.0,COSTA MARQUES,RO,120.0,116.0,-3.33,-4.0,18
4,1100098.0,ESPIGÃO D'OESTE,RO,19.0,14.0,-26.32,-5.0,0
5,1100106.0,GUAJARÁ-MIRIM,RO,143.0,130.0,-9.09,-13.0,120
6,1100122.0,JI-PARANÁ,RO,118.0,195.0,65.25,77.0,104
7,1100189.0,PIMENTA BUENO,RO,16.0,17.0,6.25,1.0,17
8,1100205.0,PORTO VELHO,RO,1445.0,1150.0,-20.42,-295.0,634
9,1100288.0,ROLIM DE MOURA,RO,32.0,51.0,59.38,19.0,41


In [51]:
# Identificando municipios exclusivos de 2026

exclusivos_2026 = set(df_2026["codigo_ibge"].dropna()) - set(comparacao["codigo_ibge"])

print("Municípios exclusivos de 2026:", len(exclusivos_2026))

Municípios exclusivos de 2026: 46


In [52]:
# Incluindo os municípios exclusios de 2026 na comparação

novos = pd.DataFrame({
    "codigo_ibge": list(exclusivos_2026)
})

novos = novos.merge(ibge_municipios, on="codigo_ibge", how="left")

novos["2024"] = 0
novos["2025"] = 0
novos["variacao_%"] = float("nan")
novos["variacao_absoluta"] = 0

novos["2026_parcial"] = novos["codigo_ibge"].map(registros_2026)

comparacao = pd.concat([comparacao, novos], ignore_index=True)

print("Municípios adicionados:", len(novos))
print("Total de municípios na comparação:", len(comparacao))

Municípios adicionados: 46
Total de municípios na comparação: 934


In [53]:
# Verificando se a soma dos registros de 2026 na tabela corresponde à quantidade original com código IBGE

print("Total original 2026:",
      df_2026["codigo_ibge"].notna().sum())

print("Total na comparação:",
      comparacao["2026_parcial"].sum())

print("Códigos IBGE duplicados:",
      comparacao["codigo_ibge"].duplicated().sum())

Total original 2026: 153078
Total na comparação: 153078
Códigos IBGE duplicados: 0


### Salvando a comparação na pasta "../data/processed/migracao" 

In [54]:
# Exporatando os indicadores 

pasta_destino = Path("../data/processed/migracao")

comparacao.to_parquet(
    pasta_destino / "indicadores_migracao_municipal.parquet",
    index=False
)

print("Indicadores salvos com sucesso!")

Indicadores salvos com sucesso!


In [55]:
# validação dos totais de 2024 e 2025

for ano, df in [(2024, df_2024), (2025, df_2025)]:
    original = df["codigo_ibge"].notna().sum()
    consolidado = comparacao[str(ano)].sum()

    print(f"{ano}:")
    print("Registros originais:", original)
    print("Registros consolidados:", int(consolidado))
    print("Diferença:", int(original - consolidado))

2024:
Registros originais: 222574
Registros consolidados: 222574
Diferença: 0
2025:
Registros originais: 236777
Registros consolidados: 236777
Diferença: 0


In [56]:
# Convertendo o código IBGE para int 

comparacao["codigo_ibge"] = comparacao["codigo_ibge"].astype("Int64")

print(comparacao["codigo_ibge"].dtype)
display(comparacao.head())

Int64


,codigo_ibge,municipio,uf,2024,2025,variacao_%,variacao_absoluta,2026_parcial
0,1100023,ARIQUEMES,RO,0.0,17.0,NaN,17.0,0
1,1100049,CACOAL,RO,94.0,89.0,-5.32,-5.0,34
2,1100056,CEREJEIRAS,RO,10.0,24.0,140.00,14.0,20
3,1100080,COSTA MARQUES,RO,120.0,116.0,-3.33,-4.0,18
4,1100098,ESPIGÃO D'OESTE,RO,19.0,14.0,-26.32,-5.0,0


In [57]:
comparacao.to_parquet(
    "../data/processed/migracao/indicadores_migracao_municipal.parquet",
    index=False
)

print("Tabela atualizada com sucesso!")

Tabela atualizada com sucesso!


In [58]:
ibge_municipios.to_parquet(
    "../data/processed/municipios_ibge.parquet", index=False
)